In [ ]:
from pathlib import Path
from dtale import show
import polars as pl
import dtale.global_state as global_state

global_state.set_app_settings(dict(max_column_width=300))

data_dir = Path().absolute() / ".." / ".." / "data"
df = pl.read_parquet(data_dir / "parsed_conversations" / "cm0i27jdj0000aqpa73ghpcxf.snappy")



In [ ]:
df = (
    df.with_columns(
        pl.concat_str(
            [
                pl.col("date"),
                pl.lit(" at "),
                pl.col("time"),
                pl.lit("----------------------------------------"),
                pl.lit("\n QUESTION: "),
                pl.col("question"),
                pl.lit("\n ANSWER: "),
                pl.col("answer"),
            ],
        ).alias("conversation_text")
    )
    .group_by("conversation_id")
    .agg(
        [
            pl.col("conversation_text").str.concat("\n").alias("conversation_text"),
            pl.col("date").first().alias("start_date"),
            pl.col("time").first().alias("start_time"),
            pl.col("date").last().alias("end_date"),
            pl.col("time").last().alias("end_time"),
            pl.col("question").count().alias("message_count"),
        ]
    )
)


In [ ]:
show(df.to_pandas()).open_browser()